# 02b · E1 con un encoder más grande: RigoBERTa Clinical

### Qué es este notebook

Repite E1 (ajuste fino de un encoder para NER) con un modelo **más grande y más nuevo**, para ver si el techo del encoder sube. Todo lo demás es igual que en el notebook 02: los mismos datos, un modelo por tipo de entidad, 3 semillas, *early stopping* con *dev*, el mismo test y las mismas métricas.

| | Encoders de E1 | RigoBERTa Clinical |
|---|---|---|
| Arquitectura | RoBERTa *base* | XLM-RoBERTa *large* |
| Capas / dimensión | 12 / 768 | 24 / 1.024 |
| Parámetros | ~125 millones | ~560 millones |
| Preentrenamiento | Clínico español (BSC) o + derivaciones chilenas | RigoBERTa 2 + ClinText-SP, el corpus clínico abierto en español más grande (2025) |

- **Fuente:** [IIC/RigoBERTa-Clinical](https://huggingface.co/IIC/RigoBERTa-Clinical), Subies et al. 2025/2026.
- **Licencia:** no comercial (rigoclinical-nc). Sirve para investigación académica; no se puede usar en un producto comercial.

### Un cambio en los hiperparámetros (declarado)

Los modelos *large* son más inestables con tasas de aprendizaje altas, así que se usa **2×10⁻⁵** en vez de 3×10⁻⁵ (el rango habitual para modelos *large* es 1–2×10⁻⁵). El resto es igual: lote de 16, hasta 5 épocas, *early stopping* con paciencia 2.

### Cuánto demora

Es unas 4 veces más lento que los encoders de E1: alrededor de 1 hora por entrenamiento en el Mac M5, es decir **unas 9 a 10 horas** para 3 tipos × 3 semillas. Conviene correrlo de noche. El orden es **semilla por semilla**: tras unas 3 horas ya están los 3 tipos con la semilla 42, y se puede mirar el resultado parcial.

### Conexión con el curso

- **U3 (*transfer learning*) y U7 (modelos fundacionales):** el efecto de la escala del modelo y del corpus de preentrenamiento.
- **U2 (entrenamiento):** la inestabilidad de los modelos grandes y por qué bajar la tasa de aprendizaje.

In [ ]:
from pathlib import Path
import sys, json, time, pickle
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml
import numpy as np
import pandas as pd
from dataclasses import asdict
from datos.cwlc import cargar_split
from modelos.e1_encoder import Config, entrenar, predecir, evaluar, guardar_resultados, dispositivo
from evaluacion import metricas
from reproducibilidad import huella_completa, verificar_huellas_datos, agregar_semillas
from sistema import mantener_despierto, dejar_dormir

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron! {difs}"
print(f"train={len(train)} dev={len(dev)} test={len(test)} | dispositivo: {dispositivo()} | huellas verificadas")

## 1. Configuración

- `MODO = "rapido"`: 200 oraciones y 1 época, solo para comprobar que el modelo se descarga (unos 2,2 GB la primera vez) y que entrena sin errores de memoria. Unos 5 minutos.
- `MODO = "completo"`: la versión que se reporta.

Si aparece un error de memoria (`MPS backend out of memory`), cambiar `BATCH = 8` y volver a correr: lo ya terminado se salta.

In [ ]:
MODO = "rapido"          # "rapido" para probar | "completo" para reportar
CLAVE = "rigoberta_clinico"
NOMBRE = cfg_global["encoders"][CLAVE]                  # IIC/RigoBERTa-Clinical
g = cfg_global["entrenamiento_encoder_grande"]
BATCH = g["batch"]
TIPOS = cfg_global["entidades"]
SEMILLAS = cfg_global["semillas"]

def config_para(tipo, semilla):
    if MODO == "rapido":
        return Config(modelo=NOMBRE, tipo=tipo, epocas=1, lr=g["lr"], batch=BATCH, max_len=g["max_len"], max_train=200, semilla=semilla)
    return Config(modelo=NOMBRE, tipo=tipo, epocas=g["epocas"], lr=g["lr"], batch=BATCH, max_len=g["max_len"], semilla=semilla)

SALIDA = RAIZ / "results" / f"e1_{MODO}"
CACHE = RAIZ / "results" / "cache" / f"e1_{MODO}"
CACHE.mkdir(parents=True, exist_ok=True)
pendientes = [(s, t) for s in SEMILLAS for t in TIPOS if not (SALIDA / f"{CLAVE}_{t}_s{s}.json").exists()]
print(MODO, NOMBRE, "| lr", g["lr"], "| lote", BATCH, f"| pendientes: {len(pendientes)} de {len(SEMILLAS) * len(TIPOS)}")

## 2. Entrenar

La celda mantiene el Mac despierto (`caffeinate`). Dejarlo **enchufado, con la tapa abierta y VS Code abierto**. Si se corta, volver a correrla: retoma donde quedó. Cada resultado guarda su huella digital (datos, código, revisión exacta del modelo en Hugging Face y entorno).

In [ ]:
despierto = mantener_despierto()
try:
    for semilla in SEMILLAS:
        for tipo in TIPOS:
            ruta = SALIDA / f"{CLAVE}_{tipo}_s{semilla}.json"
            ruta_pred = CACHE / f"{CLAVE}_{tipo}_s{semilla}.pkl"
            if ruta.exists() and ruta_pred.exists():
                print(f"(ya existe) {ruta.name}"); continue
            print(f"\n=== {CLAVE} · {tipo} · semilla {semilla} · {time.strftime('%H:%M')} ===")
            cfg = config_para(tipo, semilla)
            t0 = time.time()
            modelo, cod, historial = entrenar(cfg, train, dev)
            pred = predecir(modelo, cod, test, tipo, disp=dispositivo())
            gold = [e for o in test for e in o.entidades if e[3] == tipo]
            res = evaluar(gold, pred, test, tipo, train=train)
            res["clave"], res["minutos_total"] = CLAVE, round((time.time() - t0) / 60, 1)
            huella = huella_completa(asdict(cfg), carpeta, modelo=NOMBRE, splits={"train": train, "dev": dev, "test": test})
            guardar_resultados(ruta, cfg, historial, res, huella)
            ruta_pred.write_bytes(pickle.dumps(pred))
            print(f"  F1 estricto = {res['estricto']['f1']:.3f} · {res['minutos_total']} min")
            del modelo
finally:
    dejar_dormir(despierto)

## 3. Comparación con los encoders de E1

Media ± desviación estándar de las semillas disponibles, y comparación pareada **por semilla** contra el mejor encoder de E1 en cada entidad (mismas oraciones, *bootstrap*). La conclusión solo es robusta si se repite en las 3 semillas.

In [ ]:
filas = []
for clave in ["espanol_clinico", "chileno_clinico", CLAVE]:
    for f in sorted(SALIDA.glob(f"{clave}_*_s*.json")):
        d = json.loads(f.read_text())
        filas.append({"modelo": clave, "tipo": d["config"]["tipo"], "semilla": d["config"]["semilla"],
                      "F1 estricto": d["resultados"]["estricto"]["f1"], "F1 laxo": d["resultados"]["laxo"]["f1"],
                      "recall nuevas": d["resultados"]["menciones_nuevas"]["recall_estricto"],
                      "min": d["resultados"].get("minutos_total")})
tab = pd.DataFrame(filas)
def media_de(x):
    a = agregar_semillas(list(x)); return f"{a['media']:.3f} ± {a['de']:.3f} (n={a['n']})"
tab.groupby(["tipo", "modelo"])[["F1 estricto", "F1 laxo", "recall nuevas"]].agg(media_de)

In [ ]:
filas = []
for tipo in TIPOS:
    gold = [e for o in test for e in o.entidades if e[3] == tipo]
    medias = tab[tab.tipo == tipo].groupby("modelo")["F1 estricto"].mean()
    rival = max(["espanol_clinico", "chileno_clinico"], key=lambda k: medias.get(k, 0))
    for s in SEMILLAS:
        pa, pb = CACHE / f"{CLAVE}_{tipo}_s{s}.pkl", RAIZ / "results" / "cache" / "e1_completo" / f"{rival}_{tipo}_s{s}.pkl"
        if pa.exists() and pb.exists():
            dif, lo, hi = metricas.bootstrap_pareado(metricas.por_documento(gold, pickle.loads(pa.read_bytes()), "estricto"),
                                                     metricas.por_documento(gold, pickle.loads(pb.read_bytes()), "estricto"))
            filas.append({"tipo": tipo, "semilla": s, "rival": rival, "RigoBERTa − rival": round(dif, 3),
                          "IC95": f"[{lo:+.3f}, {hi:+.3f}]", "significativa": lo > 0 or hi < 0})
pd.DataFrame(filas)

## 4. Siguiente paso

Si RigoBERTa Clinical mejora, sus predicciones quedan en `results/cache/e1_completo/` y el **notebook 05** las incluye solo: aparecen en todas las tablas y en la votación entre modelos (sección 7).

**Para el informe:**
- Declarar el cambio de tasa de aprendizaje.
- Declarar la licencia no comercial del modelo.
- Declarar que es casi 5 veces más grande: más lento y con más memoria para usarlo en un CESFAM.